In [64]:
from build123d import (
    Align,
    Axis,
    BuildPart,
    BasePartObject,
    BuildLine,
    BuildSketch,
    Circle,
    FilletPolyline,
    GridLocations,
    MM,
    Mode,
    Polyline,
    Plane,
    RectangleRounded,
    RotationLike,
    add,
    extrude,
    export_stl,
    make_face,
    mirror,
    revolve,
)
from ocp_vscode import show
from gridfinity_build123d import BaseEqual

BASE_LENGTH = 6 # Units
BASE_WIDTH = 4 # Units
BASE_CORNER_RADIUS = 7.5 / 2 # mm
HEIGHT = 63 # mm

CHOP_LENGTH = 220 # mm
CHOP_WIDTH  = 160 # mm
CHOP_CORNER_RADIUS = 35 # mm
CHOP_HEIGHT = HEIGHT - 7 # mm

SIDE_DOUBLE_LENGTH = 75 # mm from outside edge of bin wall to edge of cutout
SIDE_HALF_LENGTH = (BASE_LENGTH * 42) / 2
SIDE_RADIUS = 12.5

In [ ]:
class ChopBin(BasePartObject):
    """Gridfinity Bin object with quirky compartment for storing IKEA chopping boards."""

    def __init__(
        self,
        height: float = 0,
        height_in_units: int = 0,
        rotation: RotationLike = (0, 0, 0),
        align: Align | tuple[Align, Align, Align] | None = None,
        mode: Mode = Mode.ADD,
    ):
        """Construct a custom bin object.

        Args:
            base (Part): Base object on which the bin is constructed.
            height (float, optional): Height of the bin in mm. Can't be used when height_in_units is
                defined.Defaults to 0.
            height_in_units (int, optional): Heigth defined by gridfinity units. Can't be used when
                height is defined. Defaults to 0.
            compartment (Compartment | None): Custom compartment of the bin, Defaults to None.
            rotation (RotationLike, optional): angles to rotate about axes. Defaults to (0, 0, 0).
            align (Union[Align, tuple[Align, Align, Align]], optional): align min, center, or max
            of object. Defaults to None.
            mode (Mode, optional): combination mode. Defaults to Mode.ADD.
        """
        if height and height_in_units:
            msg = "height or height_in_units can be defined, not both"
            raise ValueError(msg)
        if height_in_units:
            bin_height = height_in_units * 7
        else:
            bin_height = height

        with BuildPart() as bin:
            # Add the base
            add(
                BaseEqual(
                    grid_x=BASE_WIDTH, 
                    grid_y=BASE_LENGTH,
                    rotation=rotation,
                    align=align,
                    mode=mode
                )
            )
            # Add a sketch on top for the chop compartment
            with BuildSketch(bin.faces().sort_by(Axis.Z)[-1]) as chop_sketch:

                RectangleRounded(
                    height=BASE_LENGTH * 42 * MM, 
                    width=BASE_WIDTH * 42 * MM,
                    radius=BASE_CORNER_RADIUS * MM,
                    align=(Align.CENTER, Align.CENTER)
                )
                RectangleRounded(
                    height=CHOP_LENGTH * MM, 
                    width=CHOP_WIDTH * MM,
                    radius=CHOP_CORNER_RADIUS * MM,
                    mode=Mode.SUBTRACT,
                    align=(Align.CENTER, Align.CENTER)
                )
            # Extrude the bin to the specified height
            extrude(to_extrude=chop_sketch.face(), amount=bin_height)

        with BuildSketch(bin.faces().sort_by(Axis.X)[0]):
            with GridLocations(x_count=1, y_count=1, x_spacing=0, y_spacing=0, align=(Align.CENTER, Align.MAX)):
                add(
                    Circle(radius=50)
                ).locate((0, 0))
                # with BuildLine():
                #     FilletPolyline(
                #         (SIDE_HALF_LENGTH * MM, CHOP_HEIGHT * MM),
                #         (SIDE_HALF_LENGTH * MM - SIDE_DOUBLE_LENGTH * MM, CHOP_HEIGHT * MM),
                #         (SIDE_HALF_LENGTH * MM - SIDE_DOUBLE_LENGTH * MM, 0),
                #         (0, 0),
                #         radius=SIDE_RADIUS * MM,
                #     )
                #     Polyline(
                #         (0, 0),
                #         (0, CHOP_HEIGHT * MM),
                #         (SIDE_HALF_LENGTH * MM, CHOP_HEIGHT * MM),
                #     )

                # make_face()
                # mirror(about=Plane.YZ)
        # Extrude the cutout from long side of bin
        # extrude(amount=BASE_WIDTH * 42 * -1 * MM, mode=Mode.SUBTRACT)

        super().__init__(bin.part, rotation, align, mode)

chop_block = ChopBin(
    height=CHOP_HEIGHT
)
# show(chop_block)
show(
    chop_block,
    chop_block.faces().filter_by(Axis.X).sort_by(Axis.X)[0],
    chop_block.faces().filter_by(Axis.X).sort_by(Axis.X)[-1],
    colors=["yellow", "red", "blue"],
    alphas=[0.5, 0.5, 0.5],
)
# export_stl(chop_block, "chop_block.stl")

TypeError: GridLocations.__init__() missing 2 required positional arguments: 'x_spacing' and 'y_spacing'

In [68]:
SIDE_DOUBLE_LENGTH = 75 # mm from outside edge of bin wall to edge of cutout
SIDE_HALF_LENGTH = (BASE_LENGTH * 42) / 2
SIDE_RADIUS = 12.5

with BuildPart() as test_bin:
    with BuildSketch() as test_sketch:
        add(
            RectangleRounded(
                height=BASE_LENGTH * 42 * MM,
                width=BASE_WIDTH * 42 * MM,
                radius=BASE_CORNER_RADIUS * MM,
                align=(Align.CENTER, Align.CENTER),
            )
        )
    extrude(amount=70 * MM)

    side_plane = Plane(test_bin.faces().sort_by(Axis.X)[-1])
    with BuildSketch(side_plane) as side_profile:
        with GridLocations(x_count=1, y_count=1, x_spacing=0, y_spacing=0, align=(Align.CENTER, Align.MAX)):
            add(
                Circle(radius=50)
            )
    extrude(to_extrude=side_plane, amount=10 * MM, mode=Mode.SUBTRACT)

# with BuildLine():
#     FilletPolyline(
#         (SIDE_HALF_LENGTH * MM, CHOP_HEIGHT * MM),
#         (SIDE_HALF_LENGTH * MM - SIDE_DOUBLE_LENGTH * MM, CHOP_HEIGHT * MM),
#         (SIDE_HALF_LENGTH * MM - SIDE_DOUBLE_LENGTH * MM, 0),
#         (0, 0),
#         radius=SIDE_RADIUS * MM,
#     )
#     Polyline(
#         (0, 0),
#         (0, CHOP_HEIGHT * MM),
#         (SIDE_HALF_LENGTH * MM, CHOP_HEIGHT * MM),
#     )

# make_face()
# mirror(about=Plane.YZ)

show(test_bin, side_plane, colors=["green", "red"], alphas=[0.5, 0.5])

RuntimeError: extrude doesn't accept Plane, did you intend <keyword>=Plane(o=(84.00, 0.00, 35.00), x=(0.00, 1.00, 0.00), z=(1.00, -0.00, -0.00))?